In [0]:

import csv
import hashlib

from pathlib import Path
from pyspark.sql import functions as F

from validation import validate_ingestion_params, validate_required_columns, validate_file_checksum


from ingestion import (
    ingest_bts_month,
    TABLE_NAME,
    REQUIRED_COLUMNS
)


In [0]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

source_path = (
    "/Volumes/workspace/aerosignal_bronze/"
    "source_files/T_ONTIME_REPORTING.csv"
)

#Calculate the actual SHA256 of the uploaded file
with open(source_path, "rb") as file:
    source_sha256 = hashlib.file_digest(file, "sha256").hexdigest()

rows = ingest_bts_month(
    spark=spark,
    source_path=source_path,
    year=2025,
    month=1,
    source_sha256=source_sha256,
)

print(f"Successfully ingested {rows} rows")

In [0]:
display(
    spark.sql("""
        SELECT
            _source_year,
            _source_month,
            COUNT(*) AS total_rows
        FROM workspace.aerosignal_bronze.flights_raw
        GROUP BY _source_year, _source_month
        ORDER BY _source_year, _source_month
    """)
)

In [0]:
%sql
SELECT
    _source_year,
    _source_month,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT _source_sha256) AS source_versions
FROM workspace.aerosignal_bronze.flights_raw
WHERE _source_year = 2025
    AND _source_month = 1
GROUP BY _source_year, _source_month;

In [0]:
rows_before = spark.table(TABLE_NAME).count()

version_before = spark.sql(
    f" DESCRIBE HISTORY {TABLE_NAME}"
).first()["version"]


try:
    # Call ingestion with wrong month
    ingest_bts_month(
        spark=spark,
        source_path="/Volumes/workspace/aerosignal_bronze/source_files/T_ONTIME_REPORTING.csv",
        year=2025,
        month=2,
        source_sha256=source_sha256
    )
except ValueError as e:
    print("Expected error:", e)
else:
    raise AssertionError("Invalid input was accepted")

rows_after = spark.table(TABLE_NAME).count()

version_after = spark.sql(
    f" DESCRIBE HISTORY {TABLE_NAME}"
).first()["version"]

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Version before:", version_before)
print("Version after:", version_after)

assert rows_before == rows_after
assert version_before == version_after

In [0]:
rows_before = spark.table(TABLE_NAME).count()

version_before = spark.sql(
    f"DESCRIBE HISTORY {TABLE_NAME}"
).first()["version"]

try:
    ingest_bts_month(
        spark=spark,
        source_path="/Volumes/workspace/aerosignal_bronze/source_files/T_ONTIME_REPORTING.csv",
        year=2025,
        month=1,
        source_sha256="0" * 64
    )
except ValueError as e:
    print("Expected error:", e)
else:
    raise AssertionError("Invalid input was accepted")

rows_after = spark.table(TABLE_NAME).count()

version_after = spark.sql(
    f"DESCRIBE HISTORY {TABLE_NAME}"
).first()["version"]

assert rows_before == rows_after
assert version_before == version_after

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Version before", version_before)
print("Version after", version_after)

print("TEST PASSED")


In [0]:
source_path = (
    "/Volumes/workspace/aerosignal_bronze/source_files/T_ONTIME_REPORTING.csv"
)

test_path = (
    "/Volumes/workspace/aerosignal_bronze/source_files/test_missing_column.csv"
)

with open(source_path, "r", newline="", encoding="utf-8-sig") as file:
    reader = csv.DictReader(file)

    columns = [
        col for col in reader.fieldnames
        if col != "FL_DATE"
    ]

    first_record = next(reader)


with open(test_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=columns,
        extrasaction="ignore"
    )

    writer.writeheader()
    writer.writerow(first_record)

print("Test CSV created successfully")

print("FL_DATE exists:", "FL_DATE" in columns)

with open(test_path, "rb") as file:
    test_sha256 = hashlib.file_digest(
        file, "sha256"
    ).hexdigest()

print("SHA256:", test_sha256)

rows_before = spark.table(TABLE_NAME).count()

version_before = spark.sql(
    f"DESCRIBE HISTORY {TABLE_NAME}"
).first()["version"]

try:
    ingest_bts_month(
        spark=spark,
        source_path=test_path,
        year=2025,
        month=1,
        source_sha256=test_sha256
    )

except ValueError as e:
    print("Expected error:", e)

    assert "Missing columns" in str(e)
    assert "FL_DATE" in str(e)

else:
    raise AssertionError("Invalid input was accepted")

rows_after = spark.table(TABLE_NAME).count()

version_after = spark.sql(
    f"DESCRIBE HISTORY {TABLE_NAME}"
).first()["version"]

print("Rows before:", rows_before)
print("Version before:", version_before)
print("Rows after:", rows_after)
print("Version after:", version_after)

assert rows_before == rows_after
assert version_before == version_after

print("TEST PASSED")

In [0]:
def assert_ingestion_rejected(
    spark=spark,
    source_path: str,
    year: int,
    month: int,
    source_sha256: str,
    expected_error: str,
) -> None:
    """
    Verify that invalid input is rejected
    without modifying the Bronze Delta table.
    """

    rows_before = spark.table(TABLE_NAME).count()

    version_before = spark.sql(
        f"DESCRIBE HISTORY {TABLE_NAME}"
    ).first()["version"]

    try:
        ingest_bts_month(
            spark=spark,
            source_path=source_path,
            year=year,
            month=month,
            source_sha256=source_sha256
        )

    except ValueError as e:
        assert str(e) == expected_error
        print("Expected error:", e)
    else:
        raise AssertionError("Invalid input was accepted")

    rows_after = spark.table(TABLE_NAME).count()

    version_after = spark.sql(
        f"DESCRIBE HISTORY {TABLE_NAME}"
    ).first()["version"]

    print("Rows before:", rows_before)
    print("Version before:", version_before)
    print("Rows after:", rows_after)
    print("Version after:", version_after)

    assert rows_before == rows_after
    assert version_before == version_after

    print("TEST PASSED")

In [0]:
test_path = (
    "/Volumes/workspace/aerosignal_bronze/source_files/test_empty.csv"
)

with open(test_path, "rb") as file:
    test_sha256 = hashlib.file_digest(
        file, "sha256"
    ).hexdigest()

assert_ingestion_rejected(
    spark=spark,
    source_path=test_path,
    year=2025,
    month=1,
    source_sha256=test_sha256,
    expected_error="Source file is empty"
)


In [0]:
validate_ingestion_params(
    year=2025,
    month=1,
    source_sha256="a" * 64
)

print("Valid parameters: PASSED")

In [0]:
try:
    validate_ingestion_params(
        year=2025,
        month=13,
        source_sha256="a" * 64
    )
except ValueError as e:
    print("Expected error:", e)

In [0]:
import unittest
import test_validation
import importlib, validation

importlib.reload(validation)
importlib.reload(test_validation)

suite = unittest.defaultTestLoader.loadTestsFromModule(
    test_validation
)

result = unittest.TextTestRunner(verbosity=2).run(suite)

assert result.wasSuccessful()